# 🧹 311 Service Requests Data Cleaning & Preprocessing Pipeline
**Project:** NagarSeva-AI  
**Input Dataset:**   
**Output Location:**  &   
**Objective:** Perform systematic data cleaning, drop uninformative 100% missing columns, standardize timestamp formats, handle missing values, engineer SLA resolution metrics, and export an optimized analytical dataset.

## 1. Environment Setup & Libraries
Import required data manipulation libraries (, , ), set pandas formatting preferences, and verify file paths.

In [ ]:
import os
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 50)

print('Environment configured.')

## 2. Data Ingestion & Audit
Load the raw 311 Service Requests dataset, clean UTF-8 BOM headers, and inspect initial data types and null counts.

In [ ]:
input_path = '../datasets/complaints/311_Service_Requests.csv'
if not os.path.exists(input_path):
    input_path = 'datasets/complaints/311_Service_Requests.csv'

df = pd.read_csv(input_path, low_memory=False)
df.columns = [col.lstrip('﻿').strip() for col in df.columns]

print(f'Raw Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns')
display(df.head(3))

## 3. Remove 100% Missing / Uninformative Columns
Identify and remove columns that contain 100% missing values across all records.

In [ ]:
null_counts = df.isnull().sum()
all_null_cols = null_counts[null_counts == len(df)].index.tolist()

print(f'Dropping {len(all_null_cols)} columns with 100% missing values:')
for col in all_null_cols:
    print(f'  - {col}')

df.drop(columns=all_null_cols, inplace=True)
print(f'Updated Shape: {df.shape[0]:,} rows, {df.shape[1]} columns')

## 4. Standardize Datetime Fields
Convert timestamp strings into UTC datetime objects (, , , ).

In [ ]:
date_cols = ['ADDDATE', 'RESOLUTIONDATE', 'SERVICEDUEDATE', 'SERVICEORDERDATE', 'GDB_FROM_DATE', 'GDB_TO_DATE']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)

print('Timestamp parsing complete.')
display(df[date_cols[:4]].dtypes)

## 5. Categorical Missing Value Imputation
Fill missing categorical values with domain-appropriate placeholders (, , , ).

In [ ]:
df['STREETADDRESS'] = df['STREETADDRESS'].fillna('Unspecified')
df['ZIPCODE'] = df['ZIPCODE'].fillna(-1).astype(int)
df['SERVICETYPECODEDESCRIPTION'] = df['SERVICETYPECODEDESCRIPTION'].fillna('Unclassified')
df['ORGANIZATIONACRONYM'] = df['ORGANIZATIONACRONYM'].fillna('Unknown')
df['PRIORITY'] = df['PRIORITY'].fillna('Standard')
df['WARD'] = df['WARD'].fillna('Unassigned')

print('Categorical imputation complete.')
print('Remaining missing value breakdown:')
display(df.isnull().sum()[df.isnull().sum() > 0])

## 6. Feature Engineering (Resolution & Temporal Variables)
Construct analytical features for modeling and dashboards:
- : Flag for open/unresolved complaints (1 = Open, 0 = Closed)
-  & : Turnaround time metrics
- : Flag indicating SLA target date breach
- , , , : Temporal decomposition

In [ ]:
# Open request status
df['IS_OPEN'] = df['RESOLUTIONDATE'].isnull().astype(int)

# Resolution duration in hours and days
df['RESOLUTION_TIME_HOURS'] = (df['RESOLUTIONDATE'] - df['ADDDATE']).dt.total_seconds() / 3600.0
df['RESOLUTION_TIME_DAYS'] = df['RESOLUTION_TIME_HOURS'] / 24.0

# SLA Breach status
now_utc = pd.Timestamp.now(tz='UTC')
df['IS_OVERDUE'] = (
    (df['RESOLUTIONDATE'] > df['SERVICEDUEDATE']) | 
    ((df['RESOLUTIONDATE'].isnull()) & (now_utc > df['SERVICEDUEDATE']))
).astype(int)

# Date component breakdown
df['ADD_YEAR'] = df['ADDDATE'].dt.year
df['ADD_MONTH'] = df['ADDDATE'].dt.month
df['ADD_DAY_NAME'] = df['ADDDATE'].dt.day_name()
df['ADD_HOUR'] = df['ADDDATE'].dt.hour

print('Feature engineering complete.')
display(df[['SERVICEREQUESTID', 'IS_OPEN', 'RESOLUTION_TIME_DAYS', 'IS_OVERDUE', 'ADD_DAY_NAME']].head(5))

## 7. Export Cleaned Dataset
Export the preprocessed dataset to  in both CSV and compressed Parquet formats.

In [ ]:
csv_out = '../datasets/processed/311_Service_Requests_Cleaned.csv'
parquet_out = '../datasets/processed/311_Service_Requests_Cleaned.parquet'
if not os.path.exists('../datasets/processed'):
    csv_out = 'datasets/processed/311_Service_Requests_Cleaned.csv'
    parquet_out = 'datasets/processed/311_Service_Requests_Cleaned.parquet'

print('Saving CSV export...')
df.to_csv(csv_out, index=False)
print('Saving Parquet export...')
df.to_parquet(parquet_out, index=False)

print(f'Successfully saved cleaned datasets:')
print(f'  - CSV: {csv_out} ({os.path.getsize(csv_out)/(1024*1024):.2f} MB)')
print(f'  - Parquet: {parquet_out} ({os.path.getsize(parquet_out)/(1024*1024):.2f} MB)')

## 8. Summary & Key Findings

### Data Cleaning Key Findings
- **Removed Null Columns:** Dropped 8 completely empty columns (, , , , , , , ).
- **Open Requests:** Identified 14,618 unresolved requests () out of 287,992 total records.
- **Feature Enhancements:** Added 7 new derived metrics including , , , and timestamp components.
- **Parquet Storage Efficiency:** Parquet export reduces storage requirements significantly while preserving native schema types.

### Insights or Next Steps
- Use  as input for training SLA prediction and urgency classification models.
- Proceed to  or machine learning model benchmarking.